In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt


In [ ]:
Ymeta=pd.read_csv('/mnt/archgen/users/xiaowen/Kamenice/1024/Hap/mtHap/mtHap_120226.csv',names=['ID','Periodiation','Genetic_Sex','MT_Haplogroup','Haplogrep_quality','Mtcov_schmutzi']
                ,skiprows=1)
Ymeta


In [ ]:
def calculate_percentage_of_group(haplogroups, target_groups):
    total_count = len(haplogroups)
    percentages = []
    for target_group in target_groups:
        if target_group == 'H':
            target_count = sum(1 for hg in haplogroups if hg.startswith(target_group) and not hg.startswith("HV"))
        else:
            target_count = sum(1 for hg in haplogroups if hg.startswith(target_group))
        percentage = (target_count / total_count) * 100
        percentages.append(percentage)
    return pd.DataFrame({'target_groups': target_groups, 'percentages': percentages})
def main():
    # Example list of Y haplogroups
    Ymeta_LBA = Ymeta[Ymeta['Periodiation'] == 'KNC_MLBA']
    haplogroups_LBA = Ymeta_LBA['MT_Haplogroup']
    Ymeta_EIA = Ymeta[Ymeta['Periodiation'] == 'KNC_EIA']
    haplogroups_EIA = Ymeta_EIA['MT_Haplogroup']
    Ymeta_IA = Ymeta[Ymeta['Periodiation'] == 'KNC_DIA']
    haplogroups_IA = Ymeta_IA['MT_Haplogroup']

    # Calculate percentage of haplogroups for multiple target groups
    target_groups = ['R1']
    percentages_LBA_df = calculate_percentage_of_group(haplogroups_LBA, target_groups)
    percentages_EIA_df = calculate_percentage_of_group(haplogroups_EIA, target_groups)
    percentages_IA_df = calculate_percentage_of_group(haplogroups_IA, target_groups)

    # First, merge the LBA and EIA results
    combined_df = pd.merge(percentages_LBA_df, percentages_EIA_df, on='target_groups', suffixes=('_LBA', '_EIA'))

    # Then merge the result with the IA data
    combined_df = pd.merge(combined_df, percentages_IA_df, on='target_groups')

    # Rename columns for clarity
    combined_df.columns = ['target_groups', 'percentages_LBA', 'percentages_EIA', 'percentages_IA']

    # Transpose the DataFrame
    transposed_df = combined_df.set_index('target_groups').T

    print(transposed_df)

if __name__ == "__main__":
    main()


In [ ]:
Ymeta_LBA = Ymeta[Ymeta['Periodiation'] == 'KNC_MLBA']
haplogroups_LBA = Ymeta_LBA['MT_Haplogroup']
Ymeta_EIA = Ymeta[Ymeta['Periodiation'] == 'KNC_EIA']
haplogroups_EIA = Ymeta_EIA['MT_Haplogroup']
Ymeta_IA = Ymeta[Ymeta['Periodiation'] == 'KNC_DIA']
haplogroups_IA = Ymeta_IA['MT_Haplogroup']

# Calculate percentage of haplogroups for multiple target groups
target_groups = ['H','HV','I','J','K','N','R','T','U','V','W','X']
percentages_LBA_df = calculate_percentage_of_group(haplogroups_LBA, target_groups)
percentages_EIA_df = calculate_percentage_of_group(haplogroups_EIA, target_groups)
percentages_IA_df = calculate_percentage_of_group(haplogroups_IA, target_groups)

# First, merge the LBA and EIA results
combined_df = pd.merge(percentages_LBA_df, percentages_EIA_df, on='target_groups', suffixes=('_LBA', '_EIA'))

# Then merge the result with the IA data
combined_df = pd.merge(combined_df, percentages_IA_df, on='target_groups')

    
# Rename columns
combined_df.columns = ['target_groups', 'MLBA (n=19)\n1/λ=5.52', 'EIA (n=126)\n1/λ=4.61', 'DIA (n=85)\n1/λ=4.67']

# Transpose the DataFrame
transposed_df = combined_df.set_index('target_groups').T

print(transposed_df)

In [ ]:
# Check whether percentages sum to 1 (or very close due to rounding)
print("LBA sum:", percentages_LBA_df['percentages'].sum())
print("EIA sum:", percentages_EIA_df['percentages'].sum())
print("DIA sum:", percentages_IA_df['percentages'].sum())


In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(10, 16),dpi=600)
ax = fig.add_subplot(111)

# Define x values for the bars
x_values = [0.2,0.5,0.8]
print
bottom = None

# Define the width of the bars
bar_width = 0.2
import seaborn as sns

# Get a color-blind-friendly palette
# Original colorblind palette with 10 colors
colors = plt.get_cmap("tab20").colors[:15]

#colors = plt.get_cmap('tab20').colors[:12]

for i, group in enumerate(transposed_df.columns):
    if bottom is None:
        ax.bar([x +  bar_width for x in x_values], transposed_df[group], width=bar_width, color=colors[i % len(colors)],label=group)
        bottom = transposed_df[group]
    else:
        ax.bar([x +  bar_width for x in x_values], transposed_df[group], width=bar_width, bottom=bottom, color=colors[i % len(colors)],label=group)
        bottom += transposed_df[group]

plt.setp(ax.get_yticklabels(), fontsize=20)
plt.legend(loc='upper left', fontsize=30, ncol=1, bbox_to_anchor=(1, 1))  # Adjust the bbox_to_anchor parameter
ax.set_xticks([x + bar_width for x in x_values])  # Setting x-ticks
ax.set_xticklabels(transposed_df.index, fontsize=20)  # Setting x-tick labels
plt.xlabel('Period', fontsize=30)  # Replace 'X Label' with your actual label
plt.ylabel('Percentage', fontsize=30)  # Replace 'Y Label' with your actual label
plt.title('mtDNA haplogroups', fontsize=30,x=0.5,y=1.01)
fig.savefig('mtHap_120226.svg',format='svg', bbox_inches='tight')
plt.show()
